In [11]:
pip install sentence-transformers numpy scikit-learn openai transformers

In [17]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI
import os
from transformers.utils import logging

# -----------------------------
# Load users
# -----------------------------
with open("users.json", "r") as f:
    users = json.load(f)

# -----------------------------
# Convert profile → text (for embeddings)
# -----------------------------
def profile_to_text(user):

    p = user["user_profile"]

    text = f"""
    Major: {p['major']}
    Year: {p['year_of_study']}
    Age: {p['age']}
    Gender: {p['gender']}
    MBTI: {p['mbti']}
    Mood: {p['mood']}
    Fitness: {p['fitness']}

    Personality:
    Extroversion {p['personality']['extroversion']}
    Group preference {p['personality']['group_preference']}
    Energy {p['personality']['energy_level']}

    Interests: {", ".join(p['interests'])}
    Classes: {", ".join(p['class'])}
    Clubs: {", ".join(p['club'])}
    """

    return text


# -----------------------------
# Generate embeddings locally
# -----------------------------
logging.set_verbosity_error()
model = SentenceTransformer("all-MiniLM-L6-v2")

profile_texts = [profile_to_text(u) for u in users]

embeddings = model.encode(profile_texts)


# -----------------------------
# Match users via vector similarity
# -----------------------------
def find_best_matches(embeddings, users):

    similarity_matrix = cosine_similarity(embeddings)

    matches = []

    for i in range(len(users)):

        best_match_idx = None
        best_score = -1

        for j in range(len(users)):

            if i == j:
                continue

            score = similarity_matrix[i][j]

            if score > best_score:
                best_score = score
                best_match_idx = j

        matches.append((users[i]["name"], users[best_match_idx]["name"], best_score))

    return matches


matches = find_best_matches(embeddings, users)


# -----------------------------
# Gen AI explanation using free endpoint
# -----------------------------
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY", "test"),
    base_url="https://vjioo4r1vyvcozuj.us-east-2.aws.endpoints.huggingface.cloud/v1"
)

def explain_match(userA, userB):

    prompt = f"""
    Explain why these two people would make good friends.

    Person A:
    {profile_to_text(userA)}

    Person B:
    {profile_to_text(userB)}

    Focus on compatibility in:
    - interests
    - personality
    - extroversion
    - energy level
    - activities
    - mbti
    - mood
    - fitness
    - year of study
    - major
    - club
    - class
    - age
    """

    resp = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": "You analyze friendship compatibility."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=120
    )

    return resp.choices[0].message.content

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [21]:
# -----------------------------
# Run system
# -----------------------------
def main() -> float:
    for nameA, nameB, score in matches:

        userA = next(u for u in users if u["name"] == nameA)
        userB = next(u for u in users if u["name"] == nameB)

        print("\n---------------------------")
        print(f"Match: {nameA} ↔ {nameB}")
        print(f"Compatibility score: {score:.2f}")

        explanation = explain_match(userA, userB)
        print("AI Explanation:")
        print(explanation)

main()


---------------------------
Match: Alice ↔ Bob
Compatibility score: 0.95
AI Explanation:
**Why Person A and Person B Would Make Great Friends**

Below is a point‑by‑point comparison that shows how the two profiles line up across the dimensions you asked for.  Wherever the scores line up, they create natural “meeting‑points” that make it easy for a friendship to spark, deepen, and stay fun.

| Dimension | Person A | Person B |

---------------------------
Match: Bob ↔ Alice
Compatibility score: 0.95
AI Explanation:
**Why Person A and Person B Are a Natural Fit for Friendship**

Below is a point‑by‑point comparison that shows how the two profiles line up across every dimension you asked for.  Wherever the traits overlap or nicely complement each other, I note the specific way it can translate into a strong, enjoyable friendship.

---

## 1.
